# CRAFT End-to-End QA Evaluation

This notebook demonstrates how to perform question answering on CRAFT retrieval results using various language models (GPT-4o, Gemini) and compute evaluation metrics.

In [ ]:
# Import required libraries
import re
import json
import pandas as pd
from typing import List, Dict, Tuple
import os
from collections import Counter
import pickle
from tqdm import tqdm
import time

# Optional imports - install as needed
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
    print("✅ OpenAI available")
except ImportError:
    OPENAI_AVAILABLE = False
    print("❌ OpenAI not available. Install with: pip install openai")

try:
    import google.generativeai as genai
    GEMINI_AVAILABLE = True
    print("✅ Gemini available")
except ImportError:
    GEMINI_AVAILABLE = False
    print("❌ Gemini not available. Install with: pip install google-generativeai")

try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
    print("✅ Tiktoken available")
except ImportError:
    TIKTOKEN_AVAILABLE = False
    print("❌ Tiktoken not available. Install with: pip install tiktoken")

## Configuration

Set up file paths and API keys here.

In [ ]:
# File paths - UPDATE THESE TO MATCH YOUR DATA LOCATIONS
METADATA_PATH = "/path/to/nq_tables_metadata_updated.csv"
QUESTIONS_PATH = "/path/to/combined.jsonl"  # Questions file
CORPUS_PATH = "/path/to/corpus_4th_stage_final.pkl"  # CRAFT retrieval results
TOP_ROWS_PATH = "/path/to/corpus_2nd_stage_row_rerank.pkl"  # Top rows per table
ROW_DATA_PATH = "/path/to/nq_row_tables.json"  # Row data

# API Keys - REPLACE WITH YOUR ACTUAL KEYS
OPENAI_API_KEY = "your-openai-key-here"
GEMINI_API_KEYS = ["your-gemini-key-1", "your-gemini-key-2"]

# Initialize APIs
if OPENAI_AVAILABLE and OPENAI_API_KEY != "your-openai-key-here":
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    print("✅ OpenAI client initialized")
else:
    openai_client = None
    print("❌ OpenAI client not initialized")

if GEMINI_AVAILABLE and GEMINI_API_KEYS[0] != "your-gemini-key-1":
    genai.configure(api_key=GEMINI_API_KEYS[0])
    print("✅ Gemini configured")
else:
    print("❌ Gemini not configured")

# Initialize tokenizer
if TIKTOKEN_AVAILABLE:
    encoding = tiktoken.encoding_for_model("gpt-4o")
    print("✅ Tokenizer initialized")
else:
    encoding = None
    print("❌ Tokenizer not available")

## Data Loading

Load all required data files for the evaluation.

In [ ]:
def load_data():
    """Load all required data files."""
    print("Loading data files...")
    
    # Load metadata
    metadata_columns = ["index", "TableID", "Table_Title", "Table_Headers", "Table_CellValues"]
    dtypes = {
        "index": int,
        "TableID": str, 
        "Table_Title": str,
        "Table_Headers": str,
        "Table_CellValues": str
    }
    metadata = pd.read_csv(
        METADATA_PATH,
        names=metadata_columns,
        dtype=dtypes,
        skiprows=1
    )
    
    # Load questions
    with open(QUESTIONS_PATH, 'r') as f:
        questions = [json.loads(line) for line in f]
    
    # Load corpus results
    with open(CORPUS_PATH, 'rb') as f:
        corpus = pickle.load(f)
        
    # Load top rows per table
    with open(TOP_ROWS_PATH, 'rb') as f:
        top_rows_per_table = pickle.load(f)
        
    # Load row data
    with open(ROW_DATA_PATH, 'r') as f:
        row_table_data = json.load(f)
    rowid_to_data = {entry['Row Number']: entry for entry in row_table_data}
    
    print(f"✅ Loaded {len(questions)} questions, {len(metadata)} tables")
    return metadata, questions, corpus, top_rows_per_table, rowid_to_data

# Load data
metadata, questions, corpus, top_rows_per_table, rowid_to_data = load_data()

## Evaluation Functions

Core functions for computing F1 scores and other metrics.

In [ ]:
def normalize_answer(s: str) -> str:
    """Normalize answer by removing punctuation and extra spaces."""
    s = re.sub(r'[^\w\s]', '', s.lower())
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def tokenize_answer(s: str) -> List[str]:
    """Tokenize normalized answer."""
    return normalize_answer(s).split()

def compute_f1_score(generated_answer: str, ground_truth_answers: List[str]) -> float:
    """
    Compute maximum F1 score between generated answer and any ground truth.
    """
    generated_tokens = tokenize_answer(generated_answer)
    if not generated_tokens:
        return 0.0
    
    max_f1 = 0.0
    for gt_answer in ground_truth_answers:
        gt_tokens = tokenize_answer(gt_answer)
        if not gt_tokens:
            continue
        
        generated_counter = Counter(generated_tokens)
        gt_counter = Counter(gt_tokens)
        common = sum((generated_counter & gt_counter).values())
        
        precision = common / sum(generated_counter.values()) if generated_counter else 0.0
        recall = common / sum(gt_counter.values()) if gt_counter else 0.0
        
        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0
        
        max_f1 = max(max_f1, f1)
    
    return max_f1

def count_tokens(text: str, model: str = "gpt-4o") -> int:
    """Count tokens in text."""
    if model.startswith("gpt") and encoding:
        return len(encoding.encode(text))
    elif model.startswith("gemini") and GEMINI_AVAILABLE:
        try:
            model_obj = genai.GenerativeModel('gemini-2.0-flash-exp')
            return model_obj.count_tokens(text).total_tokens
        except:
            return len(text.split())  # Fallback
    else:
        return len(text.split())  # Fallback

print("✅ Evaluation functions defined")

## Table Processing Functions

Functions to extract and format table content for QA prompts.

In [ ]:
def get_table_text(qid: str, table_ids: List[str], use_mini_table: bool = True) -> Tuple[List[str], List[str]]:
    """
    Retrieve table text for given table IDs.
    
    Args:
        qid: Query ID
        table_ids: List of table IDs to retrieve
        use_mini_table: If True, use top 5 rows only; if False, use all rows
        
    Returns:
        Tuple of (table_texts, valid_table_ids)
    """
    table_texts = []
    valid_table_ids = []
    
    for table_id in table_ids:
        rows = top_rows_per_table.get(str(qid), {}).get(table_id, [])
        if not rows:
            continue
        
        row_texts = []
        for rid in rows:
            if rid in rowid_to_data:
                row_texts.append(rowid_to_data[rid]["Row Data"])
        
        if row_texts:
            if use_mini_table:
                table_text = " ".join(row_texts[:5])  # Mini-table: top 5 rows
            else:
                # Get full table from metadata
                table_data = metadata.loc[metadata["TableID"] == table_id, 
                                        ["Table_Headers", "Table_CellValues"]].fillna('').agg(' '.join, axis=1).values
                if len(table_data) > 0:
                    table_text = table_data[0].replace('|', ' ')
                else:
                    table_text = " ".join(row_texts)  # Fallback to row data
            
            table_texts.append(table_text)
            valid_table_ids.append(table_id)
    
    return table_texts, valid_table_ids

def create_prompt(question: str, table_texts: List[str], valid_table_ids: List[str]) -> str:
    """Create QA prompt for language model."""
    tables_str = "\n".join([
        f"Table Title: {metadata.loc[metadata['TableID'] == tid, 'Table_Title'].iloc[0]}, "
        f"Table Content: {content}"
        for tid, content in zip(valid_table_ids, table_texts)
    ])
    
    prompt = f"""You are given a set of tables along with their titles. Based on the information provided in the most relevant table(s) answer the following question. Do **not** include any explanations or extra text.
Ensure the final answer format is **only**: 
'FINAL ANSWER: Answer Value'

[TABLES]
{tables_str}

QUESTION: {question}
FINAL ANSWER:"""
    
    return prompt

print("✅ Table processing functions defined")

## Language Model Query Functions

Functions to query OpenAI and Gemini models.

In [ ]:
def query_openai(prompt: str, model: str = "gpt-4o") -> str:
    """Query OpenAI model."""
    if not openai_client:
        return "Error: OpenAI client not initialized"
        
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that provides concise answers."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=100,
            temperature=0.0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error querying OpenAI: {str(e)}"

def query_gemini(prompt: str, model: str = "gemini-2.0-flash-exp") -> str:
    """Query Gemini model."""
    if not GEMINI_AVAILABLE:
        return "Error: Gemini not available"
        
    try:
        model_obj = genai.GenerativeModel(model)
        response = model_obj.generate_content(
            prompt,
            generation_config={
                "max_output_tokens": 100,
                "temperature": 0.0
            }
        )
        return response.text.strip()
    except Exception as e:
        return f"Error querying Gemini: {str(e)}"

def extract_answer(response: str) -> str:
    """Extract answer from model response."""
    answer_match = re.search(r'FINAL ANSWER: (.*)', response)
    return answer_match.group(1).strip() if answer_match else response

print("✅ LM query functions defined")

## Token Efficiency Analysis

Compare token usage between mini-tables and full tables.

In [ ]:
def analyze_token_efficiency(sample_size: int = 50):
    """Analyze token efficiency of mini-tables vs full tables."""
    print(f"Analyzing token efficiency (sample size: {sample_size})")
    
    mini_table_tokens = []
    full_table_tokens = []
    
    for i, q_data in enumerate(questions[:sample_size]):
        if str(i) not in corpus:
            continue
            
        qid = q_data['qid']
        if qid not in top_rows_per_table:
            qid = qid + "_0"
            
        question = q_data['OriginalQuestion']
        table_ids = [tid for (tid, _) in corpus[str(i)][:5]]  # Top 5 tables
        
        # Get mini-table and full table versions
        mini_tables, valid_ids_mini = get_table_text(qid, table_ids, use_mini_table=True)
        full_tables, valid_ids_full = get_table_text(qid, table_ids, use_mini_table=False)
        
        if mini_tables and full_tables:
            mini_prompt = create_prompt(question, mini_tables, valid_ids_mini)
            full_prompt = create_prompt(question, full_tables, valid_ids_full)
            
            mini_tokens = count_tokens(mini_prompt)
            full_tokens = count_tokens(full_prompt)
            
            mini_table_tokens.append(mini_tokens)
            full_table_tokens.append(full_tokens)
    
    if mini_table_tokens and full_table_tokens:
        avg_mini = sum(mini_table_tokens) / len(mini_table_tokens)
        avg_full = sum(full_table_tokens) / len(full_table_tokens)
        reduction = (avg_full - avg_mini) / avg_full * 100
        
        print(f"📊 Token Analysis Results:")
        print(f"   Mini-table avg tokens: {avg_mini:.1f}")
        print(f"   Full-table avg tokens: {avg_full:.1f}")
        print(f"   Token reduction: {reduction:.1f}%")
        
        return {
            'mini_table_tokens': mini_table_tokens,
            'full_table_tokens': full_table_tokens,
            'avg_mini': avg_mini,
            'avg_full': avg_full,
            'reduction_percent': reduction
        }
    else:
        print("❌ No valid data for token analysis")
        return None

# Run token efficiency analysis
token_analysis = analyze_token_efficiency(sample_size=50)

## QA Evaluation

Run end-to-end QA evaluation on a sample of queries.

In [ ]:
def run_qa_evaluation(model_name: str = "gpt-4o", 
                     num_tables: int = 5,
                     use_mini_table: bool = True,
                     max_queries: int = 10):
    """
    Run QA evaluation on a sample of queries.
    
    Args:
        model_name: Model to use ("gpt-4o", "gemini-2.0-flash-exp")
        num_tables: Number of top tables to use
        use_mini_table: Whether to use mini-tables or full tables
        max_queries: Maximum number of queries to process
    """
    print(f"🔍 Running QA evaluation:")
    print(f"   Model: {model_name}")
    print(f"   Tables: {num_tables}")
    print(f"   Mini-table: {use_mini_table}")
    print(f"   Max queries: {max_queries}")
    
    results = []
    f1_scores = []
    token_counts = []
    
    for i, q_data in enumerate(tqdm(questions[:max_queries], desc="Processing queries")):
        if str(i) not in corpus:
            continue
            
        qid = q_data['qid']
        if qid not in top_rows_per_table:
            qid = qid + "_0"
            
        question = q_data['OriginalQuestion']
        ground_truth = q_data['AnswerTexts']
        
        # Get top N table IDs from corpus
        table_ids = [tid for (tid, _) in corpus[str(i)][:num_tables]]
        
        # Get table text
        table_texts, valid_table_ids = get_table_text(qid, table_ids, use_mini_table)
        
        if not table_texts:
            continue
        
        # Create prompt and count tokens
        prompt = create_prompt(question, table_texts, valid_table_ids)
        token_count = count_tokens(prompt, model_name)
        token_counts.append(token_count)
        
        # Query model
        if model_name.startswith("gpt"):
            response = query_openai(prompt, model_name)
        elif model_name.startswith("gemini"):
            response = query_gemini(prompt, model_name)
            time.sleep(1)  # Rate limiting
        else:
            response = f"Error: Unknown model {model_name}"
        
        # Extract and evaluate answer
        generated_answer = extract_answer(response)
        f1_score = compute_f1_score(generated_answer, ground_truth)
        f1_scores.append(f1_score)
        
        # Store result
        result = {
            'query_num': i,
            'qid': qid,
            'question': question,
            'response': response,
            'generated_answer': generated_answer,
            'ground_truth': ground_truth,
            'f1_score': f1_score,
            'token_count': token_count,
            'num_tables': num_tables,
            'use_mini_table': use_mini_table
        }
        results.append(result)
    
    # Compute summary statistics
    if f1_scores:
        avg_f1 = sum(f1_scores) / len(f1_scores)
        avg_tokens = sum(token_counts) / len(token_counts)
        
        print(f"\n📈 Evaluation Results:")
        print(f"   Queries processed: {len(results)}")
        print(f"   Average F1 Score: {avg_f1:.3f}")
        print(f"   Average tokens: {avg_tokens:.1f}")
        
        return {
            'model': model_name,
            'avg_f1': avg_f1,
            'avg_tokens': avg_tokens,
            'results': results,
            'f1_scores': f1_scores,
            'token_counts': token_counts
        }
    else:
        print("❌ No results generated")
        return None

# Example evaluations (uncomment to run)
# eval_gpt4o_mini = run_qa_evaluation("gpt-4o", num_tables=1, use_mini_table=True, max_queries=10)
# eval_gpt4o_full = run_qa_evaluation("gpt-4o", num_tables=1, use_mini_table=False, max_queries=10)

## Comparative Analysis

Compare different configurations (models, table counts, mini vs full tables).

In [ ]:
def comparative_analysis(max_queries: int = 20):
    """Run comparative analysis across different configurations."""
    print(f"🔬 Running comparative analysis (max queries: {max_queries})")
    
    configurations = [
        {"model": "gpt-4o", "tables": 1, "mini": True, "name": "GPT-4o, 1 table, mini"},
        {"model": "gpt-4o", "tables": 1, "mini": False, "name": "GPT-4o, 1 table, full"},
        {"model": "gpt-4o", "tables": 5, "mini": True, "name": "GPT-4o, 5 tables, mini"},
        {"model": "gpt-4o", "tables": 5, "mini": False, "name": "GPT-4o, 5 tables, full"},
    ]
    
    results_summary = []
    
    for config in configurations:
        print(f"\n⚙️  Testing: {config['name']}")
        
        result = run_qa_evaluation(
            model_name=config["model"],
            num_tables=config["tables"], 
            use_mini_table=config["mini"],
            max_queries=max_queries
        )
        
        if result:
            summary = {
                'config_name': config['name'],
                'model': config['model'],
                'num_tables': config['tables'],
                'use_mini_table': config['mini'],
                'avg_f1': result['avg_f1'],
                'avg_tokens': result['avg_tokens'],
                'queries_processed': len(result['results'])
            }
            results_summary.append(summary)
    
    # Display comparison table
    if results_summary:
        print(f"\n📊 Comparative Results Summary:")
        print("-" * 80)
        print(f"{'Configuration':<25} {'F1 Score':<10} {'Tokens':<10} {'Queries':<8}")
        print("-" * 80)
        
        for summary in results_summary:
            print(f"{summary['config_name']:<25} {summary['avg_f1']:<10.3f} "
                  f"{summary['avg_tokens']:<10.0f} {summary['queries_processed']:<8}")
        
        return results_summary
    else:
        print("❌ No results to compare")
        return None

# Uncomment to run comparative analysis
# comparison_results = comparative_analysis(max_queries=10)

## Usage Examples

Examples of how to use the evaluation functions.

In [ ]:
# Example 1: Single query evaluation
def example_single_query():
    """Example: Evaluate a single query."""
    print("🔍 Example: Single query evaluation")
    
    # Take the first question
    q_data = questions[0]
    qid = q_data['qid']
    if qid not in top_rows_per_table:
        qid = qid + "_0"
    
    question = q_data['OriginalQuestion'] 
    ground_truth = q_data['AnswerTexts']
    
    print(f"Question: {question}")
    print(f"Ground truth: {ground_truth}")
    
    # Get top 3 tables
    if "0" in corpus:
        table_ids = [tid for (tid, _) in corpus["0"][:3]]
        
        # Test both mini and full tables
        mini_tables, valid_ids = get_table_text(qid, table_ids, use_mini_table=True)
        full_tables, valid_ids = get_table_text(qid, table_ids, use_mini_table=False)
        
        if mini_tables and full_tables:
            mini_prompt = create_prompt(question, mini_tables, valid_ids)
            full_prompt = create_prompt(question, full_tables, valid_ids)
            
            print(f"\nMini-table tokens: {count_tokens(mini_prompt)}")
            print(f"Full-table tokens: {count_tokens(full_prompt)}")
            print(f"Token reduction: {(count_tokens(full_prompt) - count_tokens(mini_prompt))/count_tokens(full_prompt)*100:.1f}%")
            
            # Uncomment to actually query models (requires API keys)
            # if openai_client:
            #     mini_response = query_openai(mini_prompt)
            #     full_response = query_openai(full_prompt)
            #     print(f"\nMini-table answer: {extract_answer(mini_response)}")
            #     print(f"Full-table answer: {extract_answer(full_response)}")

# Run example
example_single_query()

In [ ]:
# Example 2: Batch processing with results saving
def example_batch_evaluation():
    """Example: Batch evaluation with results saving."""
    print("📦 Example: Batch evaluation")
    
    # Run evaluation on first 5 queries
    sample_results = []
    
    for i in range(min(5, len(questions))):
        if str(i) not in corpus:
            continue
            
        q_data = questions[i]
        qid = q_data['qid']
        if qid not in top_rows_per_table:
            qid = qid + "_0"
        
        question = q_data['OriginalQuestion']
        ground_truth = q_data['AnswerTexts']
        table_ids = [tid for (tid, _) in corpus[str(i)][:3]]
        
        # Process with mini-tables
        mini_tables, valid_ids = get_table_text(qid, table_ids, use_mini_table=True)
        if mini_tables:
            prompt = create_prompt(question, mini_tables, valid_ids)
            tokens = count_tokens(prompt)
            
            result = {
                'query_id': i,
                'question': question,
                'ground_truth': ground_truth,
                'table_ids': table_ids,
                'tokens': tokens,
                'prompt_length': len(prompt)
            }
            sample_results.append(result)
    
    # Save results
    with open('sample_evaluation_results.jsonl', 'w') as f:
        for result in sample_results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    
    print(f"✅ Processed {len(sample_results)} queries")
    print("📁 Results saved to 'sample_evaluation_results.jsonl'")
    
    return sample_results

# Run example
batch_results = example_batch_evaluation()

## Summary

This notebook provides a complete framework for evaluating QA performance on CRAFT retrieval results. Key features:

1. **Token Efficiency Analysis**: Compare mini-tables vs full tables
2. **Multi-Model Support**: GPT-4o, Gemini, and extensible to other models
3. **Comprehensive Metrics**: F1 score, token count, processing time
4. **Flexible Configuration**: Different numbers of tables, table types
5. **Batch Processing**: Handle large-scale evaluations
6. **Results Persistence**: Save detailed results for further analysis

To use this notebook:
1. Update the file paths in the configuration section
2. Add your API keys
3. Run the data loading section
4. Execute the evaluation functions as needed
5. Analyze results and compare configurations

The code is modular and can be easily adapted for different datasets or evaluation metrics.